# Experiment 8: Effect of Dropout Probability on Model Performance

**Objective**: Demonstrate how changing the dropout probability in a neural network affects the model's performance and its ability to generalize (addressing overfitting).

**Dataset**: MNIST (Handwritten Digits)

**Methodology**:
1. Load and preprocess the MNIST dataset.
2. Define a neural network architecture with a configurable Dropout layer.
3. Train the model with different dropout rates (e.g., 0.0, 0.2, 0.5, 0.8).
4. Compare the training and validation accuracy/loss to analyze the effect of dropout.

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")

## 2. Load and Preprocess Dataset

In [ ]:
# Load MNIST dataset
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to be between 0 and 1
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Flatten the images (28x28 -> 784)
X_train = X_train.reshape((-1, 784))
X_test = X_test.reshape((-1, 784))

# One-hot encode the labels
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

## 3. Define Model with Configurable Dropout

In [ ]:
def create_model(dropout_rate=0.0):
    """
    Creates a simple MLP model with a Dropout layer.
    
    Parameters:
    -----------
    dropout_rate : float
        The probability of dropping a unit (0.0 means no dropout).
    """
    model = keras.Sequential([
        layers.Dense(512, activation='relu', input_shape=(784,)),
        layers.Dropout(dropout_rate),
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

## 4. Train Models with Different Dropout Rates

In [ ]:
dropout_rates = [0.0, 0.2, 0.5, 0.8]
histories = {}

epochs = 15
batch_size = 128

for rate in dropout_rates:
    print(f"\nTraining model with Dropout Rate: {rate}")
    model = create_model(dropout_rate=rate)
    
    # We use validation_split to monitor overfitting on a held-out set
    history = model.fit(X_train, y_train,
                        epochs=epochs,
                        batch_size=batch_size,
                        validation_split=0.2,
                        verbose=1)
    
    histories[rate] = history.history

## 5. Analyze and Visualize Results

In [ ]:
# Plot Validation Accuracy
plt.figure(figsize=(10, 6))
for rate in dropout_rates:
    plt.plot(histories[rate]['val_accuracy'], label=f'Dropout {rate}')

plt.title('Validation Accuracy vs. Epochs for Different Dropout Rates')
plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot Validation Loss
plt.figure(figsize=(10, 6))
for rate in dropout_rates:
    plt.plot(histories[rate]['val_loss'], label=f'Dropout {rate}')

plt.title('Validation Loss vs. Epochs for Different Dropout Rates')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot Training vs Validation Accuracy for each rate to see overfitting directly
plt.figure(figsize=(14, 10))

for i, rate in enumerate(dropout_rates):
    plt.subplot(2, 2, i+1)
    plt.plot(histories[rate]['accuracy'], label='Train Accuracy')
    plt.plot(histories[rate]['val_accuracy'], label='Val Accuracy')
    plt.title(f'Dropout Rate: {rate}')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion/Observations

1.  **Dropout = 0.0 (No Dropout)**: The model likely overfits. Training accuracy will be very high (near 1.0), but validation accuracy might plateau or even decrease slightly as validation loss increases.
2.  **Dropout = 0.2 / 0.5**: These are typically good values. You should observe a smaller gap between training and validation accuracy compared to the 0.0 case, indicating better generalization.
3.  **Dropout = 0.8 (High Dropout)**: The model might underfit. Training accuracy will be lower, and validation accuracy might also be lower because too much information is being dropped during training, making it hard for the network to learn effectively. However, the gap between train and val accuracy will be very small.